[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/celoy/intro-ai-ed353/blob/main/notebooks/02a_bias_variance_regularization.ipynb)

# Overfitting, the bias–variance trade-off and regularization

In this notebook we fit polynomials of increasing degree to noisy data, and see that a model that fits the
training data better does not necessarily predict better. We then estimate the bias and the variance of the
fitted models, and use regularization (ridge regression) to trade one for the other.

The data are synthetic: we know the true function and can draw as many datasets as we want.

In [ ]:
# Run this cell first. On Google Colab it downloads the course files
# (data, helper modules); on your own computer it does nothing.
import os, sys
if "google.colab" in sys.modules:
    repo = "/content/intro-ai-ed353"
    if not os.path.isdir(repo):
        os.system(f"git clone --depth 1 -q https://github.com/celoy/intro-ai-ed353.git {repo}")
    os.chdir(f"{repo}/notebooks")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## The model problem

The data are generated by
$$y = f(x) + \varepsilon, \qquad f(x) = \sin(2\pi x), \qquad x \in [0, 1],$$
where the noise $\varepsilon$ is Gaussian with zero mean and standard deviation $\sigma = 0.3$, independent from
one measurement to the next and independent of $x$.

The training set has $N = 20$ points on a fixed, regular grid in $x$ (think of measurements repeated at the same
locations): only the noise changes from one training set to another. The test points are drawn uniformly in
$[0, 1]$.

In [ ]:
sigma = 0.3                      # noise standard deviation
N = 20                           # number of training points
x_train = np.linspace(0, 1, N)   # fixed grid of training points


def f_true(x):
    return np.sin(2 * np.pi * x)


def noisy(x, rng):
    """Noisy measurements y = f(x) + noise at the points x."""
    return f_true(x) + sigma * rng.normal(size=x.shape)


rng = np.random.default_rng(0)
y_train = noisy(x_train, rng)
x_test = rng.uniform(0, 1, 1000)
y_test = noisy(x_test, rng)

x_plot = np.linspace(0, 1, 201)
plt.plot(x_plot, f_true(x_plot), 'k--', label='$f(x)$')
plt.plot(x_train, y_train, 'ko', label='training data')
plt.xlabel('$x$')
plt.ylabel('$y$')
plt.legend()
plt.show()

## 1. Overfitting

### Polynomial regression

The hypothesis is a polynomial of degree $d$. As in the multivariate regression of notebook 01b, it is linear in
the parameters:
$$h_\theta(x) = \theta_0 + \theta_1 t + \theta_2 t^2 + \dots + \theta_d t^d, \qquad t = 2x - 1 \in [-1, 1],$$
and the features are the powers $1, t, \dots, t^d$. We use $t$ rather than $x$ so that all features have comparable
magnitudes (recall the rescaling in 01b). The function `poly_features` builds the $N \times (d+1)$ matrix $X$ of
features.

Least squares minimizes $J(\theta) = \frac{1}{2N}\|X\theta - y\|^2$. Rather than the normal equation
$\theta = (X^TX)^{-1}X^Ty$ of 01b, we use `np.linalg.lstsq`, which solves the least-squares problem directly from
$X$ (by singular value decomposition) and is numerically more robust.

In [ ]:
def poly_features(x, d):
    """Matrix of the features 1, t, t^2, ..., t^d with t = 2x - 1 (one row per point)."""
    return np.vander(2 * x - 1, d + 1, increasing=True)


def predict(theta, x):
    """Prediction of the polynomial with parameters theta at the points x."""
    return poly_features(x, len(theta) - 1) @ theta


def fit_ls(x, y, d):
    """Least-squares fit of a polynomial of degree d to the data (x, y)."""
    X = poly_features(x, d)
    theta, *_ = np.linalg.lstsq(X, y, rcond=None)
    return theta

### Question 1
- Fit polynomials of degree $d = 1, 3, 9$ and $15$ to the training data, and plot them together with the data
  and $f(x)$.
- Print the condition number of $X^TX$ (`np.linalg.cond`) for each degree. What does it imply for the normal
  equation of 01b?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_plot, f_true(x_plot), 'k--', label='$f(x)$')
ax.plot(x_train, y_train, 'ko', label='training data')
for d in [1, 3, 9, 15]:
    pass  # fit, plot the polynomial, print the condition number of X^T X

ax.set_ylim(-2, 2)
ax.set_xlabel('$x$')
ax.set_ylabel('$y$')
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.show()

### Question 2
Compute the root-mean-square (RMS) error on the training set and on the test set for $d = 0, 1, \dots, 15$, and
plot both as functions of $d$ (logarithmic scale for the error). How do the two errors vary with $d$? Which degree
would you choose, and why can the training error not be used to choose it?

In [ ]:
def rms(theta, x, y):
    """Root-mean-square error of the polynomial with parameters theta on the data (x, y)."""
    return np.sqrt(np.mean((predict(theta, x) - y)**2))


degrees = np.arange(16)
err_train = np.zeros(len(degrees))
err_test = np.zeros(len(degrees))
# complete and plot

## 2. The bias–variance decomposition

The fitted parameters depend on the training set $\mathcal{D}$, that is on the noise realization. Let
$\hat f_\mathcal{D}(x)$ be the prediction at $x$ of the model fitted on $\mathcal{D}$, and $y = f(x) + \varepsilon$
a new measurement at $x$, with noise independent of $\mathcal{D}$. Averaging over this noise and over training sets,
the expected squared error splits into three terms:
$$\mathbb{E}\big[(y - \hat f_\mathcal{D}(x))^2\big]
= \underbrace{\sigma^2}_{\text{noise}}
+ \underbrace{\big(f(x) - \mathbb{E}_\mathcal{D}[\hat f_\mathcal{D}(x)]\big)^2}_{\text{bias}^2}
+ \underbrace{\mathbb{E}_\mathcal{D}\Big[\big(\hat f_\mathcal{D}(x) - \mathbb{E}_\mathcal{D}[\hat f_\mathcal{D}(x)]\big)^2\Big]}_{\text{variance}}.$$
No model can reduce the noise term. Below, the bias$^2$ and the variance are also averaged over $x \in [0, 1]$.

Since the data are synthetic, we can estimate each term by drawing many training sets.

### Question 3
Complete the function `bias_variance`. For each of `K` training sets (same `x_train`, new noise), it fits the model,
stores its predictions on the grid `x_plot`, and computes its mean squared error on new noisy test data. Then:
- plot the bias$^2$, the variance, $\sigma^2$ + bias$^2$ + variance and the Monte Carlo test error as functions of
  $d$ (logarithmic scale);
- check that the decomposition holds;
- identify the regions of underfitting and overfitting.

*Remark:* the bias$^2$ is estimated from the mean of only $K$ predictions, which has itself a variance of about
variance$/K$. Estimates of the bias$^2$ smaller than variance$/K$ are therefore dominated by this Monte Carlo
error.

In [ ]:
def bias_variance(fit, K=500, seed=1):
    """Bias^2 and variance (averaged over x in [0, 1]) and mean squared test error of a fitting method.

    fit(x, y) must return the parameters theta fitted on the data (x, y).
    """
    rng = np.random.default_rng(seed)
    preds = np.zeros((K, len(x_plot)))   # predictions of the K fitted models on x_plot
    test_err = np.zeros(K)               # mean squared test error of each fitted model
    for k in range(K):
        y = noisy(x_train, rng)          # a new training set
        # complete: fit, predict on x_plot, mean squared error on new test data

    bias2 = ...      # complete
    variance = ...   # complete
    return bias2, variance, test_err.mean()


# bv[:, 0] is the bias^2, bv[:, 1] the variance and bv[:, 2] the test error, for each degree
bv = np.array([bias_variance(lambda x, y: fit_ls(x, y, d)) for d in degrees])
# plot

## 3. Ridge regression

Ridge regression adds to the cost a penalty on the size of the parameters, except the intercept $\theta_0$:
$$J_\lambda(\theta) = \frac{1}{2N}\Big[\|X\theta - y\|^2 + \lambda \sum_{j=1}^{d}\theta_j^2\Big].$$
Setting its gradient to zero gives
$$\theta = (X^TX + \lambda D)^{-1}X^Ty, \qquad D = \mathrm{diag}(0, 1, \dots, 1).$$
For $\lambda > 0$, the matrix $X^TX + \lambda D$ is invertible and much better conditioned than $X^TX$.
Penalizing all $\theta_j$ in the same way only makes sense if the features have comparable scales, hence again the
rescaled variable $t$.

### Question 4
- Complete `fit_ridge` (use `np.linalg.solve` rather than computing the inverse).
- For $d = 15$, plot the fits for $\lambda = 10^{-6}, 10^{-3}, 10^{-1}$ and $10$.
- Plot the norm $\|(\theta_1, \dots, \theta_{15})\|$ as a function of $\lambda$ for $10^{-8} \le \lambda \le 10^2$
  (log–log scale), and compare with the least-squares fit of Question 1.

In [ ]:
def fit_ridge(x, y, d, lam):
    """Ridge regression with a polynomial of degree d; the intercept is not penalized."""
    X = poly_features(x, d)
    # complete
    return theta


lambdas = np.logspace(-8, 2, 41)
# plot the fits and the norm of the parameters

### Question 5
Use `bias_variance` to compute the bias$^2$ and the variance of ridge regression with $d = 15$ as functions of
$\lambda$, and plot them as in Question 3. Which $\lambda$ minimizes the test error? Compare with the best
unregularized polynomial of Question 3.

In [ ]:
bv_ridge = np.array([bias_variance(lambda x, y: fit_ridge(x, y, 15, lam)) for lam in lambdas])
# plot